# Pipeline Bronze → Silver — Dataset Marcobre

## Descripción
Proceso de transformación desde la capa **Bronze** hacia **Silver**, enfocado en estandarización, limpieza y preparación de datos para consumo analítico.

## Alcance
- Lectura del dataset operacional desde Bronze.
- Normalización de valores nulos y tipos numéricos.
- Generación de `DateKey` (YYYYMMDD) y atributos temporales.
- Aplicación de cálculos operacionales requeridos.
- Carga en formato **Delta Lake** con control de esquema.

## Tabla destino
`dev_silver_curated_minsur.marcobre_fileserver.dataset_master`

In [0]:
%run ./Parametros

In [0]:
TABLE_BRONZE = f"{bronze_env}.marcobre_fileserver.dataset_master_dia"
TABLE_SILVER = f"{silver_env}.marcobre_fileserver.dataset_master_dia"

In [0]:
# Lista de identificadores de widgets
widgets = ['Dia']

# Crear widgets del notebook de Databricks
create_widgets(widgets)
param = get_widget_values(widgets)

In [0]:
dias = int(param["Dia"]) if param["Dia"] else 0

In [0]:
hoy = datetime.today().date()
fecha_corte = hoy - timedelta(days=dias)

datekey_corte = int(fecha_corte.strftime("%Y%m%d"))
datekey_corte

In [0]:
df = spark.read.table(TABLE_BRONZE)

numeric_cols = [
    field.name for field in df.schema.fields
    if isinstance(field.dataType, (DoubleType, FloatType))
]

df_clean = df.select(
    *[
        F.when(F.isnan(F.col(c)), None).otherwise(F.col(c)).alias(c)
        if c in numeric_cols else F.col(c)
        for c in df.columns
    ]
)

In [0]:

df_silver = df_clean.withColumn(
    "DateKey",
    F.date_format("fecha_data_unica", "yyyyMMdd").cast("int")
).select(
    "DateKey",
    *[c for c in df_clean.columns]
)

In [0]:
df_silver_final = df_silver.filter(F.col("DateKey") >= datekey_corte)

In [0]:
silver_vacia = False
try:
    if spark.table(TABLE_SILVER).limit(1).count() == 0:
        silver_vacia = True
except AnalysisException:
    silver_vacia = True

if silver_vacia:
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_SILVER)
else:
    schema_table = spark.table(TABLE_SILVER).schema
    for field in schema_table:
        if field.name in df_silver_final.columns:
            df_silver_final = df_silver_final.withColumn(
                field.name,
                F.col(field.name).cast(field.dataType)
            )
            
    spark.sql(f"DELETE FROM {TABLE_SILVER} WHERE DateKey >= {datekey_corte}")
    df_silver_final.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TABLE_SILVER)